# RIN — эмбриональный граф: ConceptNet + ATOMIC + Numberbatch

Пульт управления для `src/embryonic/` — собрать небольшой seeded-подграф по
`docs/graph_memory_manual/chapters/embryonic.tex` и поразведовать его: кто куда ведёт, какая
связность между ConceptNet-концептами и ATOMIC-событиями, как выглядит шум в
сырых данных.

**Статус на момент написания:** `api.conceptnet.io` второй день отдаёт 502, поэтому
ниже используется `conceptnet_backend="local_dump"` — тот же ограниченный BFS по entries/radius,
но по локальному дампу в `source/`, без сети. Когда API отойдёт — можно переключить
на `"api"` в конфиге ниже, ничего больше менять не надо.

In [1]:
# src/ активно меняется в этом сеансе — autoreload подхватывает правки
# в .py-файлах без перезапуска kernel (перезапуск всё равно нужен при
# смене сигнатур/переименованиях, но не при обычных правках тела функций)
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

def find_repo_root(marker="requirements.txt"):
    # Notebook now lives in src/experiments/, whose cwd in Jupyter is
    # usually the notebook's own directory, not the repo root — walk up
    # until a known repo-root marker is found, instead of assuming cwd.
    path = Path.cwd()
    for candidate in [path, *path.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"could not find repo root (looked for {marker})")

REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [3]:
import time
from collections import Counter

import matplotlib.pyplot as plt
import networkx as nx

from embryonic import conceptnet_source
from embryonic.loader import EmbryonicGraphLoader
from embryonic.numberbatch import Numberbatch

## Конфигурация

`ENTRIES`/`RADIUS` — точки входа и глубина расширения. `LIMIT_PER_NODE`/`MAX_NODES` —
предохранители от взрыва на хабах (см. `EmbryonicGraphLoader` в `src/embryonic/loader.py`).

In [4]:
SOURCE_DIR = REPO_ROOT / "source"
CONCEPTNET_DUMP = SOURCE_DIR / "conceptnet-assertions-5.7.0.csv.gz"
ATOMIC_CSV = SOURCE_DIR / "v4_atomic_all_agg.csv"  # агрегированный, не v4_atomic_all.csv
NUMBERBATCH_FILE = SOURCE_DIR / "numberbatch-en-19.08.txt.gz"

ENTRIES = ["Bacalhau", "Lisbon"]
RADIUS = 4
LANGUAGE = "en"
LIMIT_PER_NODE = 30   # соседей на концепт, самые тяжёлые рёбра в ConceptNet первыми
MAX_NODES = 1500       # жёсткий потолок; крутите, если граф слишком большой/маленький

## Шаг 1: индекс ConceptNet (строится один раз)

Один проход по всему англоязычному подмножеству дампа (~1.8M концептов из 34M рёбер
всего ConceptNet). Занимает ≈150 секунд на этой машине — запустите и ждите,
запускать ещё раз не надо: `adjacency` переиспользуется ниже для любого числа
экспериментов с разными `ENTRIES`/`RADIUS`, без повторного стриминга 475-МБ файла.

In [5]:
t0 = time.time()
adjacency = conceptnet_source.build_language_adjacency(str(CONCEPTNET_DUMP), LANGUAGE)
print(f"построено за {time.time() - t0:.0f}с, {len(adjacency):,} концептов ({LANGUAGE}) проиндексировано")

построено за 147с, 1,787,373 концептов (en) проиндексировано


## Шаг 2: Numberbatch (для связки ATOMIC ↔ ConceptNet)

Тоже строится один раз и переиспользуется ниже — это мультиязычный файл на 3 ГБ,
перечитывать его на каждый эксперимент не надо.

In [6]:
t0 = time.time()
numberbatch = Numberbatch()
numberbatch.load(str(NUMBERBATCH_FILE))
print(f"загружено за {time.time() - t0:.0f}с, {len(numberbatch.terms):,} терминов")

загружено за 73с, 516,782 терминов


## Шаг 3: собрать граф под текущие ENTRIES/RADIUS

Быстро благодаря переиспользованию `adjacency`/`numberbatch` — эту ячейку можно
перезапускать с другими `ENTRIES`/`RADIUS`/`MAX_NODES` выше сколько угодно, ответ
придёт практически мгновенно.

In [7]:
loader = EmbryonicGraphLoader(
    entries=ENTRIES,
    radius=RADIUS,
    language=LANGUAGE,
    conceptnet_backend="local_dump",
    conceptnet_adjacency=adjacency,
    limit_per_node=LIMIT_PER_NODE,
    max_nodes=MAX_NODES,
    include_atomic=True,
    atomic_path=str(ATOMIC_CSV),
    numberbatch=numberbatch,
)

t0 = time.time()
G = loader.load()
print(f"собрано за {time.time() - t0:.1f}с")
print(f"всего узлов: {len(G.nodes)}")

собрано за 2.9с
всего узлов: 105936


## Общая статистика

In [8]:
kind_counts = Counter(node.node_kind for node in G.nodes.values())
print("По видам узлов (node_kind):", dict(kind_counts))

degrees = Counter({node_id: G.degree(node_id) for node_id in G.nodes})
print("\nТоп-15 узлов по исходящей степени:")
for node_id, degree in degrees.most_common(15):
    node = G.nodes[node_id]
    print(f"  [{node.node_kind:8s}] {node.content!r:40s} degree={degree}")

По видам узлов (node_kind):

 {'concept': 1500, 'episode': 104436}

Топ-15 узлов по исходящей степени:
  [episode ] 'PersonX keeps PersonY company'          degree=111
  [episode ] 'PersonX plays ___ with friends'         degree=66
  [episode ] "PersonX grabs PersonY's bag"            degree=64
  [episode ] "PersonX stops PersonY's truck"          degree=60
  [episode ] "PersonX gets PersonY's car back"        degree=59
  [episode ] 'PersonX hikes with friends'             degree=59
  [episode ] 'PersonX watches a movie together'       degree=59
  [episode ] "PersonX plays ___ with PersonY's friends" degree=58
  [episode ] 'PersonX takes PersonY to work'          degree=58
  [episode ] 'PersonX gets some friends together'     degree=56
  [episode ] 'PersonX goes shopping with PersonY'     degree=56
  [episode ] "PersonX is camping with PersonX's friends" degree=56
  [episode ] 'PersonX meets some friends'             degree=56
  [episode ] 'PersonX robs the bank'                  degree=56
  [episode ] "PersonX go

## Кто куда ведёт — обход конкретного узла

In [9]:
def build_incoming_index(G):
    incoming = {node_id: [] for node_id in G.nodes}
    for node_id in G.nodes:
        for edge in G.out_edges(node_id):
            incoming.setdefault(edge.dst, []).append((node_id, edge))
    return incoming


INCOMING = build_incoming_index(G)


def find_nodes(G, substring: str):
    substring = substring.lower()
    return [n for n in G.nodes.values() if substring in n.content.lower()]


def show_edges(G, node_id: int, max_edges: int = 20):
    node = G.nodes[node_id]
    print(f"[{node_id}] {node.content!r} ({node.node_kind}, conf={node.confidence})")
    out_edges = G.out_edges(node_id)
    print(f"  -> {len(out_edges)} исходящих:")
    for edge in out_edges[:max_edges]:
        print(f"     --{edge.type}--> {G.nodes[edge.dst].content!r} (w={edge.weight:.2f})")
    incoming = INCOMING.get(node_id, [])
    print(f"  <- {len(incoming)} входящих:")
    for src_id, edge in incoming[:max_edges]:
        print(f"     {G.nodes[src_id].content!r} --{edge.type}--> сюда (w={edge.weight:.2f})")

In [10]:
from collections import Counter
kinds = Counter(n.node_kind for n in G.nodes.values())
stitch = [(s, e) for s in G.nodes for e in G.out_edges(s)
          if e.type == "associates"
          and G.nodes[s].node_kind == "episode"
          and G.nodes[e.dst].node_kind == "concept"]
print("node_kind:", dict(kinds))                 # episode == 0  -> ATOMIC не прицепился
print("стыков episode->concept:", len(stitch))   # это и есть «сшивка»

# что человеческого висит на конкретном концепте:
def human_around(sub):
    for cid, n in G.nodes.items():
        if n.node_kind == "concept" and sub in n.content.lower():
            evs = [s for s, _ in INCOMING.get(cid, []) if G.nodes[s].node_kind == "episode"]
            print(f"{n.content!r} <- ", [G.nodes[s].content for s in evs[:8]])

human_around("fish")   # на своём графе

node_kind: {'concept': 1500, 'episode': 104436}
стыков episode->concept: 3299
'bringing home fish' <-  []


In [11]:
for match in find_nodes(G, "lisbon"):
    show_edges(G, match.id)
    print()

for match in find_nodes(G, "bacalhau"):
    show_edges(G, match.id)
    print()

[2] 'lisbon' (concept, conf=0.7)


  -> 2 исходящих:
     --IsA--> 'place (n)' (w=0.50)
     --Synonym--> 'lisbon' (w=0.50)
  <- 7 входящих:
     'lisbon' --Synonym--> сюда (w=0.50)
     'nos (n)' --AtLocation--> сюда (w=0.50)
     'pide' --AtLocation--> сюда (w=0.50)
     'umm (n)' --AtLocation--> сюда (w=0.50)
     'lisboner' --DerivedFrom--> сюда (w=1.00)
     'lisbonite' --DerivedFrom--> сюда (w=1.00)
     'portugal' --capital--> сюда (w=0.50)

[6] 'lisboner' (concept, conf=0.7)
  -> 2 исходящих:
     --DerivedFrom--> 'lisbon' (w=1.00)
     --DerivedFrom--> 'lisbon (n)' (w=1.00)
  <- 2 входящих:
     'lisboners (n)' --FormOf--> сюда (w=1.00)
     'lisbonite (n)' --Synonym--> сюда (w=1.00)

[7] 'lisbonite' (concept, conf=0.7)
  -> 2 исходящих:
     --DerivedFrom--> 'lisbon' (w=1.00)
     --DerivedFrom--> 'lisbon (n)' (w=1.00)
  <- 2 входящих:
     'lisbonites (n)' --FormOf--> сюда (w=1.00)
     'lisboner (n)' --Synonym--> сюда (w=1.00)

[20] 'lisbon (n)' (concept, conf=0.7)
  -> 0 исходящих:
  <- 2 входящих:
     'l

## Визуализация маленького фрагмента

Не pygraphviz: он требует системно установленный Graphviz (`dot`) ПЛЮС С-компилятор
для сборки привязки на Windows — на этой машине `dot` не найден. Вместо него —
`networkx` + `matplotlib`: ни одного внешнего бинарника, работает сразу из коробки. Граф
целиком (до нескольких сотен узлов) рисовать бессмысленно — ниже вырезается
маленькая окрестность вокруг сидов внутри уже построенного `G` (это быстро, всё
в памяти, без обращения к дампу/API).

In [12]:
def local_neighborhood(G, incoming_index, center_ids, hops=1, max_nodes=30):
    visited = set(center_ids)
    frontier = list(center_ids)
    for _ in range(hops):
        next_frontier = []
        for node_id in frontier:
            neighbor_ids = [e.dst for e in G.out_edges(node_id)]
            neighbor_ids += [src_id for src_id, _e in incoming_index.get(node_id, [])]
            for other_id in neighbor_ids:
                if other_id not in visited and len(visited) < max_nodes:
                    visited.add(other_id)
                    next_frontier.append(other_id)
        frontier = next_frontier
    return visited


seed_ids = [n.id for n in G.nodes.values() if n.content in ("bacalhau", "lisbon")]
neighborhood = local_neighborhood(G, INCOMING, seed_ids, hops=1, max_nodes=30)

H = nx.DiGraph()
for node_id in neighborhood:
    H.add_node(node_id, label=G.nodes[node_id].content[:24])
for node_id in neighborhood:
    for edge in G.out_edges(node_id):
        if edge.dst in neighborhood:
            H.add_edge(node_id, edge.dst, label=edge.type)

for node in H.nodes:
    print(G.nodes[node])

Node(id=0, content='bacalhau', embedding=array([ 8.840e-02,  8.640e-02,  1.466e-01, -6.940e-02,  6.460e-02,
        1.738e-01,  8.420e-02, -5.330e-02,  8.300e-03, -7.780e-02,
        1.100e-02,  7.810e-02, -2.640e-02,  1.860e-01,  1.430e-02,
       -1.544e-01,  4.980e-02,  1.007e-01, -8.010e-02, -2.083e-01,
       -1.100e-01,  8.700e-02, -6.980e-02,  1.040e-02,  7.230e-02,
        7.140e-02,  1.383e-01,  3.060e-02, -7.650e-02,  5.350e-02,
       -2.770e-02, -5.400e-02, -4.730e-02,  7.990e-02,  1.150e-02,
       -1.310e-01,  1.040e-02, -5.050e-02, -1.980e-02, -4.830e-02,
        4.700e-03, -9.900e-02,  4.440e-02,  9.510e-02, -7.890e-02,
        2.230e-02,  2.670e-02, -2.280e-02,  5.010e-02, -4.490e-02,
       -7.020e-02,  5.150e-02, -2.340e-02,  6.300e-02,  8.990e-02,
       -2.400e-02, -1.360e-02,  4.680e-02, -1.100e-02, -2.180e-02,
       -9.230e-02,  1.270e-01, -4.800e-03,  4.410e-02, -9.750e-02,
       -3.840e-02, -6.290e-02, -6.600e-03,  2.500e-02,  6.060e-02,
        2.850e-02, -4

## Дальше

- **API вернётся —** поменяйте `conceptnet_backend="local_dump"` на `"api"` в ячейке с `EmbryonicGraphLoader`
  и уберите `conceptnet_adjacency=adjacency` — тогда он пойдёт через живой запрос.
- **Шум в данных —** однобуквенные/морфологические узлы (типа `"n"`, `"a"`) — ожидаемы на
  сыром k-hop expansion; `embryonic.tex` именно поэтому рекомендует фильтр по весу ребра +
  ручную чистку как следующий шаг после expansion — здесь этот шаг не сделан.
- **“Andrew Ng” как сид —** ConceptNet — база здравого смысла, не именованных сущностей; если
  у конкретного человека мало/нет рёбер — это ожидаемое поведение базы, не баг кода.
- **Схема ATOMIC —** убедитесь, что `ATOMIC_CSV` указывает на `v4_atomic_all_agg.csv`
  (агрегированный), а не `v4_atomic_all.csv` (сырой, одна строка на ответ разметчика) —
  парсер в `atomic_source.py` писан под первый.

## Retrieval Steps 1–2: query → якоря → диффузия

Два новых оператора, вместе реализующие двухшаговый retrieval из `definitions.tex`:

1. **`embryonic.query.query_vectors`** — сырой текст query → набор векторов (один на слово, без mean-pooling).
2. **`memory.retrieval.nearest_nodes_by_embedding`** — для каждого вектора, косинусно-ближайшие узлы по `node.embedding` — это якоря (Step 1, НЕ random walker).
3. **`memory.diffuse.diffuse`** — детерминированная диффузия заряда от якорей (Step 2), `a(t+1) = decay·P̃ᵀa(t) + s(t)`.

Использует тот же `G`/`nb`, что уже построен выше — перестраивать ничего не надо.

In [13]:
from embryonic.query import query_vectors
from memory.retrieval import nearest_nodes_by_embedding
from memory.diffuse import diffuse

### Шаг 1: query → якоря

In [14]:
QUERY = "what is the difference between lisbon and bacalhau"

vecs = query_vectors(QUERY, numberbatch)
print(f"{len(vecs)} слов с вектором из query")

anchor_ids = set()
for v in vecs:
    matches = nearest_nodes_by_embedding(G, v, top_k=3)
    anchor_ids.update(matches)
    print("  ->", [G.nodes[m].content for m in matches])

print("Якоря:", [(a, G.nodes[a].content) for a in anchor_ids])

4 слов с вектором из query
  -> []
  -> ['middle']
  -> ['lisbon', 'lisboner', 'lisbonite']
  -> ['bacalhau', 'bringing home fish', 'pide']
Якоря: [(0, 'bacalhau'), (1345, 'bringing home fish'), (2, 'lisbon'), (4, 'pide'), (6, 'lisboner'), (7, 'lisbonite'), (214, 'middle')]


### Шаг 2: диффузия от якорей

`restart=0` (дефолт) соответствует `α=0` retrieval-политики из `04_walker.tex` — одиноразовый импульс, без постоянного рестарта.

In [15]:
source_amplitudes = {a: 100.0 for a in anchor_ids}
lit = diffuse(G, source_amplitudes, decay=0.8, min_propagate_fraction=0.01, max_steps=20)

lit_sorted = sorted(lit.items(), key=lambda kv: -kv[1])
print(f"Засветилось {len(lit)} узлов из {len(G.nodes)}")
print(f"{'содержимое':30s} {'вид':10s} {'пик-амплитуда'}")
for nid, amp in lit_sorted[:25]:
    n = G.nodes[nid]
    print(f"{n.content!r:30s} {n.node_kind:10s} {amp:.2f}")

Засветилось 101 узлов из 105936
содержимое                     вид        пик-амплитуда
'lisbon'                       concept    146.67
'bacalhau'                     concept    100.00
'bringing home fish'           concept    100.00
'pide'                         concept    100.00
'lisboner'                     concept    100.00
'lisbonite'                    concept    100.00
'middle'                       concept    100.00
'making money'                 concept    80.00
'lisbon (n)'                   concept    80.00
'location'                     concept    80.00
'place (n)'                    concept    58.67
'government agency (n)'        concept    26.67
'administrative unit (n)'      concept    7.11
'executive branch of government (n)' concept    7.11
'government related entity (n)' concept    7.11
'social unit (n)'              concept    5.69
'legal government organization (n)' concept    5.69
'chess'                        concept    3.35
'euphemism'                    conc

### Сравнение: без рестарта (retrieval) против с рестартом (черновик будущего sleep-режима)

`restart>0` тянет к `α>0` — постоянное подпитывание якорей каждый шаг — это не для retrieval (манифест резервирует его за сном), но полезно увидеть разницу вживую:
с рестартом граф охватывается шире, тяготея к хабам графа, как в тесте #1.

In [16]:
lit_restart = diffuse(
    G, source_amplitudes, decay=0.8, min_propagate_fraction=0.01, max_steps=20, restart=0.15
)
print(f"без рестарта: {len(lit)} узлов засветилось")
print(f"с restart=0.15:  {len(lit_restart)} узлов засветилось")

top_restart = sorted(lit_restart.items(), key=lambda kv: -kv[1])[:10]
print("Топ-10 с рестартом:")
for nid, amp in top_restart:
    print(f"  {G.nodes[nid].content!r:30s} {amp:.2f}")

без рестарта: 101 узлов засветилось
с restart=0.15:  101 узлов засветилось
Топ-10 с рестартом:
  'lisbon'                       161.67
  'bacalhau'                     100.00
  'bringing home fish'           100.00
  'pide'                         100.00
  'lisboner'                     100.00
  'lisbonite'                    100.00
  'middle'                       100.00
  'making money'                 80.00
  'lisbon (n)'                   80.00
  'location'                     80.00


## Retrieval Step 4: сборщик (context-packer)

`memory.packer.pack_context` берёт выход диффузии и отбирает узлы в промпт по токен-бюджету (не по фиксированному числу узлов), с эпистемическими метками:

- `prior` — эмбриональный контент (ConceptNet/ATOMIC): родовое знание, не личное наблюдение Рин
- `observed`/`hypothesis` — зарезервированы под реальные наблюдения/fusion (пока не построены — ветка спит, не мёртва)
- `low_conf` — |confidence| < 0.3 (зона незнания из `definitions.tex`)

Пакер возвращает структуру (`PackedNode`, провенанс всегда внутри, для аудита) — `to_text()` ниже это только плоский inspector-вид для чтения глазами, а не финальный промпт-рендер.

In [17]:
from memory.packer import pack_context, to_text

packed = pack_context(G, lit, token_budget=200)
print(f"{len(packed)} узлов вошло в бюджет (token_budget=200)")
print(to_text(packed))

17 узлов вошло в бюджет (token_budget=200)
[prior] lisbon: IsA place (n), Synonym lisbon
[prior] bacalhau
[prior] bringing home fish: UsedFor making money
[prior] pide: AtLocation lisbon, IsA government agency (n), Synonym pide
[prior] lisboner: DerivedFrom lisbon, DerivedFrom lisbon (n)
[prior] lisbonite: DerivedFrom lisbon, DerivedFrom lisbon (n)
[prior] middle: RelatedTo location
[prior] making money
[prior] lisbon (n)
[prior] location
[prior] place (n): Synonym location, HasContext chess, HasContext euphemism, HasContext horse racing, HasContext physical
[prior] government agency (n): IsA administrative unit (n), IsA executive branch of government (n), IsA government related entity (n)
[prior] administrative unit (n): IsA social unit (n)
[prior] executive branch of government (n): IsA legal government organization (n)
[prior] government related entity (n)
[prior] social unit (n)
[prior] legal government organization (n): IsA government related entity (n)


### Структура всегда доступна, даже если `to_text()` что-то скрывает (например, провенанс для `prior`)

In [18]:
sample = packed[0]
print("node_id:", sample.node_id)
print("content:", sample.content)
print("labels:", sample.labels)
print("activation:", sample.activation)
print("edges:", sample.edges)
print("provenance (всегда в структуре, даже если не в тексте):", sample.provenance)

node_id: 2
content: lisbon
labels: ['prior']
activation: 146.66666666666666
edges: ['IsA place (n)', 'Synonym lisbon']
provenance (всегда в структуре, даже если не в тексте): ['/c/en/lisbon']


### Бюджет, а не фиксированное число узлов — узкий бюджет режет раньше, не добивая хвостом из дальней периферии

In [19]:
for budget in (20, 100, 200, 800):
    n = len(pack_context(G, lit, token_budget=budget))
    print(f"token_budget={budget:4d} -> {n} узлов")

token_budget=  20 -> 2 узлов
token_budget= 100 -> 10 узлов
token_budget= 200 -> 17 узлов
token_budget= 800 -> 101 узлов
